# 4.7 章节实践：动态 Residual + LayerNorm + GELU 近似

本节是第四章的综合实践，用一个融合算子检查中高级内容的掌握情况。实践目标不是引入新 API，而是把前面几节的能力组合起来：Residual Connection、LayerNorm、GELU 近似、动态 batch、`pypto.loop`、`pypto.view`、`valid_shape`、`pypto.assemble` 和 PyTorch reference 验证。

本节练习的计算目标是：

```text
residual = x + residual_input
mean = mean(residual, dim=-1)
var = mean((residual - mean)^2, dim=-1)
norm = (residual - mean) / sqrt(var + eps)
output = gelu_approx(norm * gamma + beta)
```


## 1. 实践目标

完成本节后，读者应能够：

1. 把一个模型模块拆成逐元素、规约、广播和写回几个步骤。
2. 使用动态 batch 描述输入，并用 `loop -> view -> compute -> assemble` 分块处理。
3. 解释 LayerNorm 中 `mean`、`var`、`gamma`、`beta` 和 `eps` 的作用。
4. 在融合算子中加入 GELU 近似，并用 PyTorch reference 检查结果。


## 2. 环境准备

先运行下面的环境准备单元。没有可用 NPU 时，本节会保留代码阅读路径，并跳过真实设备验证。


In [ ]:
import os
import torch
import pypto

try:
    import torch_npu
except ImportError:
    torch_npu = None


def reset_pypto_notebook_state():
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass


def get_device():
    if torch_npu is None:
        print("torch_npu is not available; NPU execution will be skipped.")
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    try:
        torch.npu.set_device(device_id)
    except Exception as exc:
        print("NPU device is not ready:", exc)
        return "cpu"
    return f"npu:{device_id}"


reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))
print("device:", device)
print("run_mode:", RUN_MODE)


## 3. 先把计算拆开

这个综合算子由四类能力组成：

| 步骤 | PyPTO 能力 | 作用 |
| --- | --- | --- |
| Residual | elementwise add | 把主分支和残差分支相加 |
| LayerNorm | `sum`、`sqrt`、广播 | 沿最后一维做归一化 |
| GELU 近似 | elementwise activation | 使用 `x * sigmoid(1.702 * x)` 做非线性变换 |
| Dynamic batch | `loop`、`view`、`valid_shape`、`assemble` | batch 维运行时变化时仍能分块处理 |

实践时不要一次性写完。先确认输入输出 shape，再写一个 batch tile 内的计算，最后把结果 assemble 回完整输出。


## 4. 章节实践题

题型包含选择题、填空题和编程题。建议先独立完成，再执行下一单元查看参考答案。

1. （选择题）本节综合编程题主要覆盖哪些能力？  
   A. Residual、LayerNorm、GELU 近似、动态 batch、loop/view/assemble 和验证闭环  
   B. 只覆盖字符串处理  
   C. 只覆盖图片读写  
   D. 只覆盖 Python 列表排序

2. （填空题）动态 batch 分块处理中，`valid_shape` 的作用是描述当前 tile 中________。

3. （选择题）LayerNorm 中 `keepdim=True` 的主要作用是什么？  
   A. 保留被规约的维度，方便 `mean` 和 `var` 广播回原始 shape  
   B. 删除 hidden 维度  
   C. 强制输出为 int32  
   D. 跳过方差计算

4. （填空题）本节的 PyTorch reference 应与 kernel 中的公式保持一致：先计算 `x + residual_input`，再做________，最后做________。

5. （选择题）`pypto.assemble(result, [b_offset, 0], out)` 表示什么？  
   A. 把当前 batch tile 的结果写回输出 Tensor 的对应位置  
   B. 删除输出 Tensor  
   C. 把输出 Tensor 转成字符串  
   D. 改变 NPU 设备编号

6. （编程题）补全 `dynamic_residual_norm_gelu_kernel`：输入 `x`、`residual_input`、`gamma`、`beta` 和输出 `out`，其中 `x/residual_input/out` 的 shape 为 `[batch, 128]`，batch 是动态维度。要求：  
   - 使用 `pypto.DYNAMIC` 描述动态 batch。  
   - 使用 `pypto.loop` 按 batch tile 分块。  
   - 使用 `pypto.view(..., valid_shape=...)` 取当前 tile。  
   - 在 tile 内完成 `Residual + LayerNorm + GELU 近似`。  
   - 使用 `pypto.assemble` 写回输出，并用 PyTorch reference 验证。


## 5. 查看答案

执行以下代码获取参考答案。


In [ ]:
!cat ./answer/04.07_answer.py


## 6. 本章小结

第四章从单个组合算子走向模型模块和系统能力：激活函数与 Softmax 训练公式拆解能力，LayerNorm/RMSNorm/FFN 训练归一化与矩阵乘组合能力，动态 shape 和控制流训练分块写回能力，Attention/Transformer 训练多 Tensor 数据流组织能力，Cost Model/ACLGraph 帮助理解系统分析与加速。章节实践把这些能力重新收束到一个可验证的融合算子中，作为进入更复杂项目实践前的检查点。
